Test the connection in the notebook

In [9]:
from dotenv import dotenv_values
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

config = dotenv_values(".env")
db_string = config["DB_STRING"]
print("Host:", make_url(db_string).host)

engine = create_engine(db_string)
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM diabetes.patient")).scalar())

Host: ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com
768


In [10]:
import pandas as pd

SCHEMA = "diabetes"
patient          = pd.read_sql_table("patient",          engine, schema=SCHEMA)
blood_metrics    = pd.read_sql_table("blood_metrics",    engine, schema=SCHEMA)
pedigree_outcome = pd.read_sql_table("pedigree_outcome", engine, schema=SCHEMA)
skin             = pd.read_sql_table("skin",             engine, schema=SCHEMA)

In [11]:
import os
os.makedirs("data", exist_ok=True)

patient.to_csv("data/patient.csv", index=False)
blood_metrics.to_csv("data/blood_metrics.csv", index=False)
pedigree_outcome.to_csv("data/pedigree_outcome.csv", index=False)
skin.to_csv("data/skin.csv", index=False)

In [12]:
patient          = pd.read_csv("data/patient.csv")
blood_metrics    = pd.read_csv("data/blood_metrics.csv", parse_dates=["measurement_date"])
pedigree_outcome = pd.read_csv("data/pedigree_outcome.csv")
skin             = pd.read_csv("data/skin.csv")

In [13]:
patient.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           768 non-null    int64  
 1   Age          768 non-null    int64  
 2   pregnancies  768 non-null    int64  
 3   bmi          768 non-null    float64
dtypes: float64(1), int64(3)
memory usage: 24.1 KB


In [14]:
blood_metrics.info()

<class 'pandas.DataFrame'>
RangeIndex: 1771 entries, 0 to 1770
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   insulin           1771 non-null   int64         
 1   glucose           1771 non-null   int64         
 2   bloodpressure     1771 non-null   int64         
 3   patientid         1771 non-null   int64         
 4   id                1771 non-null   int64         
 5   measurement_date  1771 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(5)
memory usage: 83.1 KB


In [15]:
import pandas as pd
from sqlalchemy import text

dataframes = {
    "patient": patient,
    "blood_metrics": blood_metrics,
    "pedigree_outcome": pedigree_outcome,
    "skin": skin,
}

def db_count(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).scalar()

for table, df in dataframes.items():
    t = f'diabetes."{table}"'

    # Rows and columns
    db_rows = db_count(f"SELECT COUNT(*) FROM {t}")
    db_cols = db_count(f"""SELECT COUNT(*) FROM information_schema.columns
                           WHERE table_schema='diabetes' AND table_name='{table}'""")
    print(f"\n===== {table} =====")
    print(f"Rows:    db = {db_rows}   notebook = {df.shape[0]}")
    print(f"Columns: db = {db_cols}   notebook = {df.shape[1]}")

    # NULL/NaN and zeros per column
    rows = []
    for col in df.columns:
        db_null = db_count(f'SELECT COUNT(*) FROM {t} WHERE "{col}" IS NULL')
        nb_null = df[col].isna().sum()

        if pd.api.types.is_numeric_dtype(df[col]):
            db_zero = db_count(f'SELECT COUNT(*) FROM {t} WHERE "{col}" = 0')
            nb_zero = (df[col] == 0).sum()
        else:
            db_zero = nb_zero = "-"   # e.g. the date column

        rows.append([col, db_null, nb_null, db_zero, nb_zero])

    print(pd.DataFrame(rows, columns=["column", "null_db", "null_nb", "zero_db", "zero_nb"])
          .to_string(index=False))


===== patient =====
Rows:    db = 768   notebook = 768
Columns: db = 4   notebook = 4
     column  null_db  null_nb  zero_db  zero_nb
         id        0        0        0        0
        Age        0        0        0        0
pregnancies        0        0      111      111
        bmi        0        0       11       11

===== blood_metrics =====
Rows:    db = 1771   notebook = 1771
Columns: db = 6   notebook = 6
          column  null_db  null_nb zero_db zero_nb
         insulin        0        0     380     380
         glucose        0        0       9       9
   bloodpressure        0        0      36      36
       patientid        0        0       0       0
              id        0        0       0       0
measurement_date        0        0       -       -

===== pedigree_outcome =====
Rows:    db = 768   notebook = 768
Columns: db = 4   notebook = 4
                  column  null_db  null_nb  zero_db  zero_nb
diabetespedigreefunction        0        0        0        0
   